# 🧠 NeuroVision AI — Train ResNet-18 on Brain Tumor MRI

This Colab notebook trains the model used by the **NeuroVision AI** Streamlit app.

**Runtime:** set `Runtime → Change runtime type → GPU (T4)`. Training takes ~10–15 min.

**Steps**
1. Download the Kaggle *Brain Tumor MRI Dataset* (4 classes).
2. Fine-tune a pretrained ResNet-18.
3. Evaluate (accuracy, macro-F1, ROC-AUC) and save `best_model.pth` + `metrics.json`.
4. Download the files and drop them into your repo (`weights/` and `assets/`).


## 1. Install & imports

In [ ]:
!pip -q install torch torchvision scikit-learn kaggle
import os, json, copy, numpy as np, torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 2. Get the dataset

**Option A (Kaggle API):** upload your `kaggle.json` (Account → Create New API Token) when prompted, then run the download cell.

Dataset: `masoudnickparvar/brain-tumor-mri-dataset` — it already contains `Training/` and `Testing/` folders with 4 class subfolders.

In [ ]:
from google.colab import files
print('Upload your kaggle.json'); files.upload()
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset -q
!unzip -q -o brain-tumor-mri-dataset.zip -d data
DATA_DIR = 'data'
print(os.listdir(DATA_DIR))

**Option B (manual):** download the dataset from Kaggle in your browser, unzip, and upload the `Training` and `Testing` folders to Colab under `data/`. Then set `DATA_DIR = 'data'`.

## 3. Data loaders

In [ ]:
IMG=224
mean=[0.485,0.456,0.406]; std=[0.229,0.224,0.225]
train_tf = transforms.Compose([
    transforms.Resize((IMG,IMG)), transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(12), transforms.ToTensor(),
    transforms.Normalize(mean,std)])
eval_tf = transforms.Compose([
    transforms.Resize((IMG,IMG)), transforms.ToTensor(),
    transforms.Normalize(mean,std)])

train_ds = datasets.ImageFolder(f'{DATA_DIR}/Training', transform=train_tf)
test_ds  = datasets.ImageFolder(f'{DATA_DIR}/Testing',  transform=eval_tf)
CLASS_NAMES = train_ds.classes
print('Classes:', CLASS_NAMES)   # ['glioma','meningioma','notumor','pituitary']

train_dl = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
test_dl  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=2)

## 4. Build the model (ResNet-18, transfer learning)

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, len(CLASS_NAMES))
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

## 5. Train

In [ ]:
EPOCHS = 6
best_f1, best_state = 0.0, None
for epoch in range(EPOCHS):
    model.train(); running=0.0
    for x,y in train_dl:
        x,y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward(); optimizer.step()
        running += loss.item()*x.size(0)
    # validation on test set
    model.eval(); preds=[]; gts=[]
    with torch.no_grad():
        for x,y in test_dl:
            out = model(x.to(device))
            preds += out.argmax(1).cpu().tolist(); gts += y.tolist()
    f1 = f1_score(gts, preds, average='macro')
    acc = accuracy_score(gts, preds)
    print(f'Epoch {epoch+1}/{EPOCHS}  loss={running/len(train_ds):.4f}  acc={acc:.4f}  macroF1={f1:.4f}')
    if f1 > best_f1:
        best_f1 = f1; best_state = copy.deepcopy(model.state_dict())
model.load_state_dict(best_state)
print('Best macro-F1:', best_f1)

## 6. Evaluate & save

In [ ]:
model.eval(); probs=[]; preds=[]; gts=[]
with torch.no_grad():
    for x,y in test_dl:
        p = torch.softmax(model(x.to(device)),1).cpu().numpy()
        probs += p.tolist(); preds += p.argmax(1).tolist(); gts += y.tolist()
probs=np.array(probs)
acc = accuracy_score(gts,preds)
f1  = f1_score(gts,preds,average='macro')
try:
    auc = roc_auc_score(gts, probs, multi_class='ovr', average='macro')
except Exception:
    auc = float('nan')
metrics = {'Accuracy':round(float(acc),3),'Macro F1':round(float(f1),3),
           'ROC-AUC (macro)':round(float(auc),3),
           'Test set':'Kaggle Brain Tumor MRI (held-out)'}
print(metrics)

os.makedirs('weights',exist_ok=True); os.makedirs('assets',exist_ok=True)
torch.save(model.state_dict(),'weights/best_model.pth')
with open('assets/metrics.json','w') as f: json.dump(metrics,f,indent=2)
print('Saved weights/best_model.pth and assets/metrics.json')

## 7. Download the files

Drop `best_model.pth` into your repo's `weights/` folder and `metrics.json` into `assets/`.

In [ ]:
from google.colab import files
files.download('weights/best_model.pth')
files.download('assets/metrics.json')